23070521054 - GAURI GULHANE

In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# 1. Dataset representing flight booking queries
sentences = [
    "book a flight from mumbai to delhi",
    "book a ticket from nagpur to pune",
    "find a flight from delhi to bangalore",
    "travel from mumbai to chennai",
    "flight from pune to hyderabad",
    "book flight from chennai to mumbai",
    "i want to fly from nagpur to delhi",
    "find ticket from bangalore to mumbai",
    "book a flight from hyderabad to pune",
    "travel from delhi to chennai",
    "i need a flight from pune to delhi",
    "book ticket from mumbai to nagpur",
    "find flight from chennai to bangalore",
    "travel from hyderabad to mumbai",
    "book a flight from delhi to pune",
    "i want to travel from nagpur to mumbai",
    "find a ticket from pune to chennai",
    "book flight from bangalore to delhi",
    "travel from mumbai to hyderabad",
    "find flight from chennai to pune"
]

# 2. Slot labels associated with each token/word
labels = [
    "O O O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O B-from_city O B-to_city",
    "O O O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O O O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city"
]

# 3. Tokenization & Numericalization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
X = tokenizer.texts_to_sequences(sentences)

# 4. Map text labels to ID values
label_to_id = {
    "O": 0,
    "B-from_city": 1,
    "B-to_city": 2
}

y = []
for label_sentence in labels:
    label_ids = [label_to_id[label] for label in label_sentence.split()]
    y.append(label_ids)

# 5. Sequence padding to standardize lengths
max_len = max(len(seq) for seq in X)
X = pad_sequences(X, maxlen=max_len, padding='post')
y = pad_sequences(y, maxlen=max_len, padding='post', value=0)

# 6. Splitting Dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Model Construction
vocab_size = len(tokenizer.word_index) + 1
num_labels = len(label_to_id)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32),
    LSTM(64, return_sequences=True),
    Dense(num_labels, activation='softmax')
])

# 8. Model Compilation
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 9. Model Training
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=4,
    validation_split=0.2,
    verbose=1
)

# 10. Evaluation
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {accuracy:.4f}")

# 11. Predicting on new sample
test_sentence = "book a flight from mumbai to pune"
test_sequence = tokenizer.texts_to_sequences([test_sentence])
test_sequence = pad_sequences(test_sequence, maxlen=max_len, padding='post')

prediction = model.predict(test_sequence, verbose=0)
predicted_ids = np.argmax(prediction, axis=-1)[0]

id_to_label = {v: k for k, v in label_to_id.items()}
words = test_sentence.split()

print("\nSlot Filling Result:")
for word, label_id in zip(words, predicted_ids):
    print(f"{word} -> {id_to_label[label_id]}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 184ms/step - accuracy: 0.6979 - loss: 1.0901 - val_accuracy: 0.7500 - val_loss: 1.0819
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7500 - loss: 1.0700 - val_accuracy: 0.7500 - val_loss: 1.0656
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.7500 - loss: 1.0462 - val_accuracy: 0.7500 - val_loss: 1.0465
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7500 - loss: 1.0166 - val_accuracy: 0.7500 - val_loss: 1.0241
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.7500 - loss: 0.9815 - val_accuracy: 0.7500 - val_loss: 0.9982
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.7500 - loss: 0.9365 - val_accuracy: 0.7500 - val_loss: 0.9709
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.7500 - loss: 0.8857 - val_accuracy: 0.7500 - val_loss: 0.9501
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7500 - loss: 0.8414 - val_accuracy: 0.7500 - val_loss